In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(
api_key=api_key
)

In [2]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages = [{'role':'user','content':'서울 현재 기온 알려줄래'}]
)

print(response.choices[0].message.content)

죄송하지만, 현재 날씨 정보를 실시간으로 제공할 수는 없습니다. 하지만 서울의 현재 기온을 확인하려면 기상청 웹사이트나 날씨 관련 앱을 이용하시면 정확한 정보를 얻으실 수 있습니다. 도움이 필요하시면 언제든지 말씀해 주세요!


## 1. 위도와 경도를 통해 날씨 정보를 얻는 함수
- 기상청 데이터 및 API(예: Open-Meteo, ECMWF, ERA5 등)
- 표준 관측 높이: 세계기상기구(WMO) 표준에 따라 지열의 직접적인 영향 및 미세한 지표면 변동을 줄이고 사람의 체감에 가까운 기온을 측정하기 위해 2미터 높이를 기준으로 합니다.

- 단위: API 및 데이터셋에 따라 일반적으로 섭씨(°C) 또는 켈빈(K)을 사용합니다.

In [3]:
import requests

# 위도 latitude, 경도 longitude
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    #print(response)
    #print(dir(response))
    data = response.json()
    print(data)
    print('data:',data['current'] )
    return data['current']['temperature_2m']  # 기온 (temperature_2m): 21.1°C
    # return data['current']['temperature_2m'], data['current']['time']

In [4]:
# GPT에 전달할 Tool(함수) 스키마 정의
# get_weather(위도, 경도)

temp = get_weather(37.5642135, 127.0016985)

# 추출한 기온 값을 소수점 이하 첫째 자리까지 서식홯하여 출력합니다.
print(f'서울 현재 기온: {temp:.1f}°C"')

{'latitude': 37.55, 'longitude': 127.0, 'generationtime_ms': 24.33490753173828, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 32.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-09-10T06:00', 'interval': 900, 'temperature_2m': 21.9, 'wind_speed_10m': 3.6}, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h'}, 'hourly': {'time': ['2026-09-10T00:00', '2026-09-10T01:00', '2026-09-10T02:00', '2026-09-10T03:00', '2026-09-10T04:00', '2026-09-10T05:00', '2026-09-10T06:00', '2026-09-10T07:00', '2026-09-10T08:00', '2026-09-10T09:00', '2026-09-10T10:00', '2026-09-10T11:00', '2026-09-10T12:00', '2026-09-10T13:00', '2026-09-10T14:00', '2026-09-10T15:00', '2026-09-10T16:00', '2026-09-10T17:00', '2026-09-10T18:00', '2026-09-10T19:00', '2026-09-10T20:00', '2026-09-10T21:00', '2026-09-10T22:00', '2026-09-10T2

In [5]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

import openmeteo_requests
import requests_cache
from retry_requests import retry

# 1. OpenAI 클라이언트 생성
# client = OpenAI()

# 2. 날씨 정보 조회 함수 정의 (Open-Meteo API 사용)
# 위도 latitude, 경도 longitude
def get_weather(latitude, longitude):

    # Open-Meteo API 전용 SDK 라이브러리와 캐싱 및 재시도 기능을 위한 세션 객체를 설정합니다.
    # 캐시 세션: 동일한 API 요청이 반복될 경우 서버 요청 없이 '.cache' 파일에 저장된 데이터를 사용합니다 (유효기간: 1시간).
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # 재시도 세션: 네트워크 에러나 서버 오류 발생 시 최대 5회까지 자동 재시도하며, 시도 간격을 지수적으로 늘립니다 (backoff_factor: 0.2초).
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # 설정한 캐시 및 재시도 정책이 적용된 Open-Meteo 전용 클라이언트 객체를 생성합니다.
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # 데이터를 요청할 Open-Meteo 기상 예보 API의 엔드포인트 URL을 지정합니다.
    url = "https://api.open-meteo.com/v1/forecast"

    # API 요청 시 전달할 쿼리 파라미터를 딕셔너리 형태로 구성합니다.
    params = {
        # 조회할 위치의 위도(서울 좌표: 37.5665)를 지정합니다.
        "latitude": latitude,
        # 조회할 위치의 경도(서울 좌표: 126.9780)를 지정합니다.
        "longitude": longitude,
        # 실시간으로 가져올 기상 항목 목록을 설정합니다 (2m 높이 기온, 2m 높이 상대습도).
        "current": ["temperature_2m", "relative_humidity_2m"],
        # 시간대 기준을 한국 표준시(KST)로 설정합니다.
        "timezone": "Asia/Seoul"
    }
    
    response = openmeteo.weather_api(url, params=params)[0]   

    # 응답 객체에서 현재 기상 정보 데이터(Current) 블록을 추출합니다.
    current = response.Current()

    # 현재 기온 (temperature_2m: 인덱스 0) 추출하여 반환
    temp = current.Variables(0).Value()
    return temp



In [7]:
# 3. GPT에 전달할 Tool(함수) 스키마 정의

# OpenAI API 모델에 전달할 Tool(함수 호출) 정의 리스트입니다.
wtool = [
    {
        # 이 도구가 실행 가능한 함수(function) 형태임을 명시합니다.
        "type": "function",
        "function": {
            # GPT가 호출할 파이썬 함수 이름을 지정합니다.
            "name": "get_weather",
            # 모델이 어떤 상황에 이 함수를 호출할지 판단하는 설명(프롬프트)입니다.
            "description": "Get current temperature for provided coordinates in celsius.",
            # 함수 실행 시 전달할 매개변수(인자)의 구조를 JSON Schema 형태로 정의합니다.
            "parameters": {
                # 매개변수 집합의 기본 데이터 타입을 객체(Object/Dict)로 지정합니다.
                "type": "object",
                # 함수에 전달될 각 인자의 속성을 정의합니다.
                "properties": {
                    # 위도 값 속성이며, 데이터 타입은 숫자(number)입니다.
                    "latitude": {"type": "number"},
                    # 경도 값 속성이며, 데이터 타입은 숫자(number)입니다.
                    "longitude": {"type": "number"},
                },
                # 함수 호출 시 반드시 전달되어야 하는 필수 인자 목록입니다.
                "required": ["latitude", "longitude"],
                # 스키마에 정의되지 않은 추가 인자의 생성을 금지합니다 (Strict Mode 필수 조건).
                "additionalProperties": False,
            },
            # 정의한 스키마 구조를 100% 엄격하게 준수하여 인자를 생성하도록 설정합니다.
            "strict": True,
        },
    }
]

In [ ]:

import json

# 4. GPT API 호출 (Function Calling 유도)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "현재 제주도 날씨는?"}],    # 현재 제주도 날씨는?
    tools=wtool,
)

if response.choices[0].message.tool_calls:
    # 5. Tool Call 결과 확인 및 Arguments 파싱
    tool_call = response.choices[0].message.tool_calls[0]
    raw_args = tool_call.function.arguments

    print("GPT가 반환한 arguments (str):", raw_args)
    print("타입:", type(raw_args))

    # JSON 문자열 -> Python dict 변환
    args = json.loads(raw_args)
    print("파싱된 args (dict):", args)
    print("타입:", type(args))

    # 6. 추출된 위도/경도로 실제 날씨 함수 실행
    c = get_weather(args["latitude"], args["longitude"])
    print(f"현재 기온: {c:.1f}°C")
else:
    print(f"GPT 일반응답: {response.choices[0].message.content}")

GPT 일반응답: 대한민국의 수도는 서울입니다.


## 2. 두 수의 합을 계산

In [15]:
def add_number(num1,num2):
    print('call', num1, num2)
    return {'합':num1+num2}

In [16]:
add_number(5,3)

call 5 3


{'합': 8}

In [ ]:
# OpenAI API 모델에 전달할 Tool(함수 호출) 정의 리스트입니다.
sum_tool = [{
    # 이 도구가 실행 가능한 함수(function) 형태임을 명시합니다.
    'type': 'function',
    'function': {
        # GPT가 호출할 파이썬 함수 이름을 지정합니다.
        'name': 'add_number',
        # 모델이 어떤 요청에서 이 함수를 사용할지 판단하는 설명입니다.
        'description': '두 숫자의 합을 계산합니다.',
        # 함수 실행 시 전달할 매개변수(인자)의 구조를 JSON Schema 형태로 정의합니다.
        'parameters': {
            # 매개변수 집합의 기본 데이터 타입을 객체(Object/Dict)로 지정합니다.
            'type': 'object',
            # 함수에 전달될 각 인자의 속성을 정의합니다.
            'properties': {
                # 첫 번째 덧셈 대상 숫자 속성이며, 데이터 타입은 숫자(number)입니다.
                'num1': {'type': 'number'},
                # 두 번째 덧셈 대상 숫자 속성이며, 데이터 타입은 숫자(number)입니다.
                'num2': {'type': 'number'}
            },
            # 함수 호출 시 반드시 전달되어야 하는 필수 인자 목록입니다.
            'required': ['num1', 'num2']
        }
    }
}]



In [18]:


messages=[{'role':'user','content':'5와 3을 더해 주세요'}]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages= messages,
     tools=sum_tool )

print( response.choices[0].message.tool_calls[0].function.arguments )
args = json.loads( response.choices[0].message.tool_calls[0].function.arguments )
result = add_number( args['num1'], args['num2'] )
# result {'합':8}
messages.append( {'role':'function','name':'add_number','content':json.dumps(result)} )
# messages=[{'role':'user','content':'5와 3을 더해 주세요'},
# {'role':'function','name':'add_number','content':"{'합':8}" } ]
response =client.chat.completions.create( model='gpt-3.5-turbo',messages= messages )
print( response.choices[0].message.content)

{"num1":5,"num2":3}
call 5 3
5와 3을 더하면 8입니다.


In [19]:
import pytz
from datetime import datetime
# tz = pytz.timezone('Asia/Seoul')
tz = pytz.timezone('Asia/Tokyo')
local_time = datetime.now(tz)
local_time.strftime('시간:%Y-%m-%d %H:%M:%S')

'시간:2026-09-10 16:30:19'

In [25]:
def get_time(city):
    timezones={
        '서울': 'Asia/Seoul',
        '도쿄': 'Asia/Tokyo',
        '뉴욕': 'America/New_York',
        '런던': 'Europe/London'
    }
    try:
        tz = pytz.timezone(timezones[city])
        local_time = datetime.now(tz)
        return local_time.strftime('시간:%Y-%m-%d %H:%M:%S')
    except Exception as err:
        return "현재 정보로는 시간을 알 수 없습니다."

In [26]:
get_time('뉴욕')

'시간:2026-09-10 03:35:50'

5. wikipedia

| 함수                                      | 설명                                      |
| --------------------------------------- | --------------------------------------- |
| `wikipedia.search(query)`               | 키워드 검색 결과(문서 제목 리스트) 반환                 |
| `wikipedia.summary(title, sentences=n)` | 해당 문서의 요약을 n문장 반환                       |
| `wikipedia.page(title)`                 | 문서 전체 페이지 객체 반환 (title, url, content 등) |
| `wikipedia.set_lang("ko")`              | 검색 언어 설정 (예: 한국어 `"ko"`, 영어 `"en"`)     |


In [ ]:
```
import wikipedia

# 위키피디아가 User-Agent 없는 요청을 차단하기 때문에 먼저 설정해줘야 함
# 주의: HTTP 헤더는 영문/숫자만 허용 (한글 넣으면 UnicodeEncodeError 발생)
wikipedia.set_user_agent('llm-workspace-edu-bot/1.0 (contact: example@example.com)')

# 검색
wikipedia.search("Python programming")

# 요약 가져오기
#  sentences=1 → 첫 문장만 반환,sentences=2 → 앞에서 2문장 반환, sentences=0 → 전체 요약 반환 (문장 제한 없음)
print(wikipedia.summary("Python (programming language)", sentences=2))

# 언어 변경 (기본: 영어)
wikipedia.set_lang("ko")  # 한국어 위키피디아 사용
print(wikipedia.summary("인공지능", sentences=5))

# 원문 페이지 가져오기
page = wikipedia.page("인공지능")
print('******')
print(page)
print(dir(page))
print('******')
print(page.title)      # 제목
print(page.url)        # URL
print(page.content[:500])  # 본문 일부
```

Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다.
인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신경망을 사용하는 기계 학습의 한 갈래이고, 생성형 인공지능은 글·그림·음성·영상·코드 같은 새로운 결과물을 만든다.
******
<WikipediaPage '인공지능'>
['_WikipediaPage__continued_query', '_WikipediaPage__load', '_WikipediaPage__title_query_param', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__g

In [2]:
import wikipedia

def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        print('search_result:', search_result)
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}

wikipedia_search("세종대왕은 누구야?" )

search_result: 대왕 세종


{'summary': '《대왕 세종(大王世宗)》은 2008년 1월 5일에서부터 2008년 11월 16일까지 방송되었던 KBS 2TV 대하 드라마 작품이다.\n\n\n== 트리비아 ==\n조선 태종 시대부터 세종 시대를 대부분의 배경으로 하고 있다. 제1회 방송 시작 하루 전인 2008년 1월 4일에 제작 뒷이야기를 다룬 스페셜 방송이 있었으며, 2008년 11월 16일까지 총 86부작으로 방송되었다.\n당초 KBS 드라마 PD인 이성주가 담당 PD로 낙점되었으나, KBS 드라마 2팀장직으로 발령되면서 연출자가 바뀌었다.\n아울러 2007년 12월 첫 회가 나갈 예정이었지만 전작 《대조영》의 연장에 따라 2008년 1월로 첫 방송일이 변경되었다.'}

In [5]:
import json
import wikipedia
from openai import OpenAI


def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}


wikifunc = [{
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "입력된 질문에 대해 필요하다면 위키피디아에서  정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": " 주제 또는 질문"
                }
            },
            "required": ["question"]
        }
    }
}]


messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=wikifunc,
    tool_choice="auto"
)


tool_call = response.choices[0].message.tool_calls[0]
print(tool_call)
args = json.loads(tool_call.function.arguments)
print( args)
result = wikipedia_search(args["question"])
print(result)

# messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"},
#             {"role": "function","name": "wikipedia_search",
#              "content":'{summany:위키피디아답변}'}]

messages.append({
    "role": "function",
    "name": "wikipedia_search",
    "content": json.dumps(result)
})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0
)


print('--------------')
print(response.choices[0].message.content)


ChatCompletionMessageFunctionToolCall(id='call_tWOdpzd8B1o5HwmC5dL0sKcZ', function=Function(arguments='{"question":"세종대왕"}', name='wikipedia_search'), type='function')
{'question': '세종대왕'}
{'summary': '세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으로 불리며 왕실과 민간에서 사용되다가 20세기 주시경이 한글로 발전시켜 오늘날 대한민국과 조선민주주의인민공화국(남북한)의 공식 문자로서 널리 쓰이고 있다.'}
--------------
세종대왕(세종, 1397년 5월 15일 ~ 1450년 3월 30일)은 조선의 제4대 왕으로, 1418년부터 1450년까지 재위했습니다. 그는 한글의 창제와 같은 중요한 업적을 남긴 왕으로, 한국 역사에서 가장 존경받는 인물 중 하나입니다.

세종대왕은 과학, 기술, 문화, 예술 등 다양한 분야에서 많은 발전을 이끌어냈습니다. 그는 농업과 천문학, 의학 등 여러 분야의 연구를 장려하고, 백성의 삶을 개선하기 위한 정책을 추진했습니다. 특히, 한글(훈민정음)의 창제는 그의 가장 큰 업적 중 하나로, 이를 통해 모든 백성이 쉽게 읽고 쓸 수 있는 기회를 제공했습니다.

그의 통치 기간 동안에는 과학기술이 크게 발전하였고, 여러 가지